In [1]:
import os
import boto3
from pyspark.sql import SparkSession

# MinIO Config
MINIO_ACCESS_KEY = "minioadmin"      # Matches docker-compose
MINIO_SECRET_KEY = "minioadmin"      # Matches docker-compose
MINIO_ENDPOINT = "http://minio:9000"

# 1. Use Boto3 to create the bucket first (Spark can't create buckets, only objects)
def create_bucket(bucket_name):
    s3 = boto3.resource('s3',
                        endpoint_url=MINIO_ENDPOINT,
                        aws_access_key_id=MINIO_ACCESS_KEY,
                        aws_secret_access_key=MINIO_SECRET_KEY,
                        config=boto3.session.Config(signature_version='s3v4'))
    if s3.Bucket(bucket_name) not in s3.buckets.all():
        s3.create_bucket(Bucket=bucket_name)
        print(f"Created bucket: {bucket_name}")

create_bucket("datalake")

# 2. Configure Spark
# Note: We don't need to add .config("spark.jars", ...) because we did it in docker-compose PYSPARK_SUBMIT_ARGS!
spark = SparkSession.builder \
    .appName("MinIOTest") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()


print("Spark Connected!")

# 3. Write Data
data = [("Apple", 10), ("Banana", 20)]
df = spark.createDataFrame(data, ["Fruit", "Quantity"])

print("Writing to MinIO...")
df.write.mode("overwrite").csv("s3a://datalake/test-data")
print("Write Success!")

# 4. Read Data
print("Reading from MinIO...")
df_read = spark.read.csv("s3a://datalake/test-data")
df_read.show()

spark.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/21 16:08:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Connected!
Writing to MinIO...


25/11/21 16:08:31 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/11/21 16:08:33 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


Write Success!
Reading from MinIO...
+------+---+
|   _c0|_c1|
+------+---+
|Banana| 20|
| Apple| 10|
+------+---+

